# Example 2: Hyperparameter Tuning and Pipeline
In this example, we train a model on the water dataset using an automated workflow. We will load the user settings from `config.yaml`, utilize `HyperOptimizer` to tune our hyperparameters, and leverage the `Pipeline` class to seamlessly handle preprocessing and model training.

First, we load the user configuration from our YAML file. To avoid writing a massive configuration file, we merge our specific settings with the package's default configuration, which automatically fills in any missing parameters.

In [7]:
import yaml

from pes_nn_fit.utils.helpers import load_default_config, merge_config

# Load user configuration
with open("config.yaml") as f:
    user_config = yaml.safe_load(f)    

print("User Configuration:")
print(yaml.dump(user_config))

# Load default and merge
default = load_default_config()
config = merge_config(default, user_config)

print("Merged Configuration:")
print(yaml.dump(config))

User Configuration:
data:
  data_file: ../../data/water.csv
  data_type: pes-learn
dataset:
  split_type: fps
hyperparams:
  model:
    layers:
      choices:
      - 8,8
      - 16,16
      dist: categorical
  preprocess:
    canonicalize_geom: true
    molecule: h2o
    morse_param:
      dist: float
      high: 1
      log: true
      low: 0.001
    morse_transform: true
    pip_transform: true
  trainer:
    adam:
      max_epoch: 100
      train: true
    early_stopping:
      enabled: true
    lm:
      lr:
        dist: float
        high: 1
        log: true
        low: 0.001
      max_epoch: 100
      train: true
optuna:
  n_trials: 10
  timeout: 1800

Merged Configuration:
data:
  data_file: ../../data/water.csv
  data_type: pes-learn
  model_file: models/model.pt
  testset_file: data/testset.npy
  trainset_file: data/trainset.npy
  valset_file: data/valset.npy
dataset:
  split_ratios:
  - 0.7
  - 0.15
  - 0.15
  split_type: fps
hyperparams:
  model:
    activation: tanh
   

Next, we load the dataset based on the configuration settings. If the input data is in Z-matrix format, we convert it into interatomic distances before splitting the data into training, validation, and testing sets.

In [8]:
from pes_nn_fit.dataset import Dataset

cfg_data = config["data"]

# Load dataset
dataset = Dataset(
    file_location=cfg_data["data_file"], 
    file_type=cfg_data["data_type"]
)

molecule = config["hyperparams"]["preprocess"]["molecule"]
dataset.zmatrix2interatom(molecule=molecule)

# Split dataset, we can select split method and ratios of the split
train_size, val_size, test_size = config["dataset"]["split_ratios"]
trainset, valset, testset = dataset.split(
    split_type=config["dataset"]["split_type"],
    train_size=train_size,
    val_size=val_size,
    test_size=test_size,
)

print(f"Train size: {trainset.n_samples}, Val size: {valset.n_samples}, Test size: {testset.n_samples}")

Train size: 693, Val size: 148, Test size: 149


We now use `HyperOptimizer` to conduct a hyperparameter search (via Optuna) on the training and validation sets. Once the study finishes, we extract the optimal parameters and merge them into our base configuration.

In [9]:
from pes_nn_fit.hypersearch import HyperOptimizer
from pes_nn_fit.utils.helpers import unflatten_params

# Run hyperparameter optimization
optimizer = HyperOptimizer(trainset, valset, config)
study = optimizer.optimize()

# Extract and merge the best hyperparameters
best_params = unflatten_params(study.best_params) # Optuna returns best parameters as flat dictionary 
best_config = merge_config(config["hyperparams"], best_params)

[I 2026-10-29 15:38:09,971] A new study created in memory with name: study
[I 2026-10-29 15:38:10,790] Trial 0 finished with value: 3.4893086169543654e-05 and parameters: {'preprocess.morse_param': 0.013292918943162165, 'model.layers': '8,8', 'trainer.lm.lr': 0.06251373574521749}. Best is trial 0 with value: 3.4893086169543654e-05.
[I 2026-10-29 15:38:12,009] Trial 1 finished with value: 0.002191556008793267 and parameters: {'preprocess.morse_param': 0.0029380279387035343, 'model.layers': '8,8', 'trainer.lm.lr': 0.39676050770529875}. Best is trial 0 with value: 3.4893086169543654e-05.
[I 2026-10-29 15:38:12,533] Trial 2 finished with value: 0.0012842211638186048 and parameters: {'preprocess.morse_param': 0.06358358856676251, 'model.layers': '8,8', 'trainer.lm.lr': 0.8123245085588685}. Best is trial 0 with value: 3.4893086169543654e-05.
[I 2026-10-29 15:38:12,803] Trial 3 finished with value: 0.8455651351776887 and parameters: {'preprocess.morse_param': 0.31428808908401085, 'model.layer

With the optimal hyperparameters identified, we combine the training and validation sets into a single comprehensive dataset. We then initialize the `Pipeline`—which automatically orchestrates preprocessing, model creation, and training—and fit it to the full training data.

In [10]:
from pes_nn_fit.model import Pipeline

# Combine train and val sets for the final training run
trainset_full = trainset.combine(valset)

# Initialize and fit the pipeline with best parameters found
pipeline = Pipeline(best_config)
pipeline.fit(trainset_full, testset)

Finally, we evaluate the trained model on the unseen test set. The pipeline automatically scales the inputs, makes energy predictions, and inverse-scales the outputs, allowing us to calculate the final Root Mean Squared Error (RMSE).

In [11]:
import numpy as np
from sklearn.metrics import mean_squared_error

# Make predictions on the test set
predictions = pipeline.predict(testset.data)

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(testset.target, predictions))

print(f"Best Hyperparameters: {study.best_params}")
print(f"Best Validation MSE during tuning: {study.best_trial.value:.2e}")
print(f"Final Test RMSE: {rmse:.2e} Hartree")

Best Hyperparameters: {'preprocess.morse_param': 0.013292918943162165, 'model.layers': '8,8', 'trainer.lm.lr': 0.06251373574521749}
Best Validation MSE during tuning: 3.49e-05
Final Test RMSE: 1.92e-03 Hartree
